## Lista de Exercícios 17/04/2026 - Versão CPLEX / Docplex

In [1]:
from docplex.mp.model import Model

#### Ex1: Problema de Dimensionamento de Frota (Fleet Sizing Problem)

In [2]:
m = Model(name="ex1-TransporteDeProdutos")

VEICULOS = ('A', 'B')
CUSTO_COMBUSTIVEL = (110, 75)

x = m.integer_var_dict(VEICULOS, lb=0, name="qtd_viagens_com")
m.minimize(
    m.sum(CUSTO_COMBUSTIVEL[i] * x[v] for i, v in enumerate(VEICULOS))
)

In [3]:
TIPOS = ('Refrigerado', 'Não Refrigerado')
CAPACIDADE_VEICULO = ((20, 30), (20, 10))
DEMANDA_PRODUTO = (160, 120)

for tipo in range(len(TIPOS)):
    m.add_constraint(
        m.sum(
            CAPACIDADE_VEICULO[i][tipo] * x[v]
            for i, v in enumerate(VEICULOS)
        ) >= DEMANDA_PRODUTO[tipo],
        ctname=f"capacidade_tipo_{tipo}",
    )

m.add_constraint(x[VEICULOS[1]] >= 4, ctname="regra_B")

docplex.mp.LinearConstraint[regra_B](qtd_viagens_com_B,GE,4)

In [4]:
m.export_as_lp(path="model1_CPLEX.lp")

'model1_CPLEX.lp'

In [5]:
solution = m.solve(log_output=True)

Version identifier: 22.1.1.0 | 2022-11-27 | 9160aff4d
CPXPARAM_Read_DataCheck                          1
Found incumbent of value 1180.000000 after 0.00 sec. (0.00 ticks)
Tried aggregator 1 time.
MIP Presolve eliminated 1 rows and 0 columns.
MIP Presolve modified 2 coefficients.
Reduced MIP has 2 rows, 2 columns, and 4 nonzeros.
Reduced MIP has 0 binaries, 2 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.00 sec. (0.00 ticks)
Tried aggregator 1 time.
Reduced MIP has 2 rows, 2 columns, and 4 nonzeros.
Reduced MIP has 0 binaries, 2 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.00 sec. (0.00 ticks)
MIP emphasis: balance optimality and feasibility.
MIP search method: dynamic search.
Parallel mode: deterministic, using up to 8 threads.
Root relaxation solution time = 0.00 sec. (0.00 ticks)

        Nodes                                         Cuts/
   Node  Left     Objective  IInf  Best Integer    Best Bound    ItCnt     Gap

*     0+    0                         1180.0000  

In [6]:
if solution:
    print(f"\nMenor consumo de combustível: {m.objective_value} L")
    for v in VEICULOS:
        print(f"Viagens com caminhão {v}: {int(round(x[v].solution_value))}")
else:
    print("Solução ótima não encontrada.")


Menor consumo de combustível: 670.0 L
Viagens com caminhão A: 2
Viagens com caminhão B: 6


#### Ex2: Problema da Mochila Multidimensional (Multidimensional Knapsack Problem)

1. Índices

In [7]:
PORAO = ('1-Proa', '2-Centro', '3-Popa')
CARGA = ('Ouro', 'Pedras Preciosas')

CAPACIDADE = (
    (2000, 1100),  # 1-Proa
    (3100, 1400),  # 2-Centro
    (1500, 600),   # 3-Popa
)
DENSIDADE = (2800, 1800)        # unidade: Kg/m³
PRECO_KG = (10000, 5000)        # unidade: Dolares/Kg
DISPONIBILIDADE_TON = (5000, 2000)  # unidade: Toneladas

In [8]:
m = Model(name="Carregamento_Navio")

x = m.continuous_var_dict(
    ((c, p) for c in CARGA for p in PORAO),
    lb=0.0,
    name="x",
)

In [9]:
m.maximize(
    m.sum(
        PRECO_KG[i] * 1000 * x[c, p]
        for i, c in enumerate(CARGA)
        for p in PORAO
    )
)

In [10]:
for i, c in enumerate(CARGA):
    m.add_constraint(
        m.sum(x[c, p] for p in PORAO) <= DISPONIBILIDADE_TON[i],
        ctname=f"disponibilidade_carga_{i}",
    )

for j, p in enumerate(PORAO):
    m.add_constraint(
        m.sum(x[c, p] for c in CARGA) <= CAPACIDADE[j][0],
        ctname=f"limite_peso_porao_{j}",
    )

for j, p in enumerate(PORAO):
    m.add_constraint(
        m.sum((x[c, p] * 1000) / DENSIDADE[i] for i, c in enumerate(CARGA))
        <= CAPACIDADE[j][1],
        ctname=f"limite_volume_porao_{j}",
    )

In [ ]:
peso_total_porao = {p: m.sum(x[c, p] for c in CARGA) for p in PORAO}

m.add_constraint(
    peso_total_porao[PORAO[0]] * CAPACIDADE[1][0] == peso_total_porao[PORAO[1]] * CAPACIDADE[0][0],
    ctname="equilibrio_proa_centro",
)

docplex.mp.LinearConstraint[equilibrio_proa_centro](3100x_Ouro_1-Proa+3100x_Pedras Preciosas_1-Proa,EQ,2000x_Ouro_2-Centro+2000x_Pedras Preciosas_2-Centro)

In [ ]:
m.add_constraint(
    peso_total_porao[PORAO[1]] * CAPACIDADE[2][0] == peso_total_porao[PORAO[2]] * CAPACIDADE[1][0],
    ctname="equilibrio_centro_popa",
)

docplex.mp.LinearConstraint[equilibrio_centro_popa](1500x_Ouro_2-Centro+1500x_Pedras Preciosas_2-Centro,EQ,3100x_Ouro_3-Popa+3100x_Pedras Preciosas_3-Popa)

In [13]:
m.export_as_lp(path="model2_CPLEX.lp")

'model2_CPLEX.lp'

In [14]:
solution = m.solve(log_output=True)

Version identifier: 22.1.1.0 | 2022-11-27 | 9160aff4d
CPXPARAM_Read_DataCheck                          1
Tried aggregator 1 time.
No LP presolve or aggregator reductions.
Presolve time = 0.02 sec. (0.01 ticks)

Iteration log . . .
Iteration:     1   Scaled dual infeas =      15000000.000000
Iteration:     3   Dual objective     =   60000000000.000000


In [15]:
if solution:
    print(f"FATURAMENTO MÁXIMO: ${m.objective_value:,.2f}")
    print("=" * 40)

    for j, p in enumerate(PORAO):
        print(f"\n> Porão: {p}")
        peso_porao = 0
        volume_porao = 0

        for i, c in enumerate(CARGA):
            qtd_ton = x[c, p].solution_value
            if qtd_ton > 0.01:
                vol_m3 = (qtd_ton * 1000) / DENSIDADE[i]
                print(f"  - {c}: {qtd_ton:.2f} Ton ({vol_m3:.2f} m³)")
                peso_porao += qtd_ton
                volume_porao += vol_m3

        pct_peso = (peso_porao / CAPACIDADE[j][0]) * 100
        pct_vol = (volume_porao / CAPACIDADE[j][1]) * 100
        print(
            f"  * Ocupação de Peso:   {peso_porao:.2f} / "
            f"{CAPACIDADE[j][0]} Ton ({pct_peso:.2f}%)"
        )
        print(
            f"  * Ocupação de Volume: {volume_porao:.2f} / "
            f"{CAPACIDADE[j][1]} m³ ({pct_vol:.2f}%)"
        )
else:
    print("Solução ótima não encontrada.")

FATURAMENTO MÁXIMO: $58,000,000,000.00

> Porão: 1-Proa
  - Ouro: 400.00 Ton (142.86 m³)
  - Pedras Preciosas: 1600.00 Ton (888.89 m³)
  * Ocupação de Peso:   2000.00 / 2000 Ton (100.00%)
  * Ocupação de Volume: 1031.75 / 1100 m³ (93.80%)

> Porão: 2-Centro
  - Ouro: 3100.00 Ton (1107.14 m³)
  * Ocupação de Peso:   3100.00 / 3100 Ton (100.00%)
  * Ocupação de Volume: 1107.14 / 1400 m³ (79.08%)

> Porão: 3-Popa
  - Ouro: 1500.00 Ton (535.71 m³)
  * Ocupação de Peso:   1500.00 / 1500 Ton (100.00%)
  * Ocupação de Volume: 535.71 / 600 m³ (89.29%)


#### Ex3: Problema de Empacotamento (Packing Problem)

- dimensão das chapas: 1,40 x 0,50 metros

In [16]:
m = Model(name="ex3-CorteDeChapasMetalicas")

MATRIZES = [1, 2, 3, 4]
x = m.integer_var_dict(MATRIZES, lb=0, name="matriz")

obj = 1 * x[1] + 2 * x[2] + 3 * x[3] + 2 * x[4]
m.minimize(obj)

In [17]:
m.add_constraint(8 * x[1] + 4 * x[2] + 2 * x[3] >= 500, ctname="corte_medio")
m.add_constraint(1 * x[2] + 2 * x[3] + 3 * x[4] >= 350, ctname="corte_grande")

docplex.mp.LinearConstraint[corte_grande](matriz_2+2matriz_3+3matriz_4,GE,350)

In [18]:
m.export_as_lp(path="model3_CPLEX.lp")

'model3_CPLEX.lp'

In [19]:
solution = m.solve(log_output=True)

Version identifier: 22.1.1.0 | 2022-11-27 | 9160aff4d
CPXPARAM_Read_DataCheck                          1
Found incumbent of value 763.000000 after 0.00 sec. (0.00 ticks)
Tried aggregator 1 time.
MIP Presolve modified 1 coefficients.
Reduced MIP has 2 rows, 4 columns, and 6 nonzeros.
Reduced MIP has 0 binaries, 4 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.00 sec. (0.00 ticks)
Tried aggregator 1 time.
Reduced MIP has 2 rows, 4 columns, and 6 nonzeros.
Reduced MIP has 0 binaries, 4 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.00 sec. (0.00 ticks)
MIP emphasis: balance optimality and feasibility.
MIP search method: dynamic search.
Parallel mode: deterministic, using up to 8 threads.
Root relaxation solution time = 0.00 sec. (0.00 ticks)

        Nodes                                         Cuts/
   Node  Left     Objective  IInf  Best Integer    Best Bound    ItCnt     Gap

*     0+    0                          763.0000        0.0000           100.00%
      0     0   

In [20]:
if solution:
    print(f"\nCusto Total Mínimo: {m.objective_value}")
    print("-" * 30)
    for i in MATRIZES:
        if x[i].solution_value > 0:
            print(f"Matriz {i}: {int(round(x[i].solution_value))} chapas")

    tot_media = 8 * x[1].solution_value + 4 * x[2].solution_value + 2 * x[3].solution_value
    tot_grande = 1 * x[2].solution_value + 2 * x[3].solution_value + 3 * x[4].solution_value
    print(f"Total produzido: {int(round(tot_media))} médias e {int(round(tot_grande))} grandes")
else:
    print("Solução ótima não encontrada.")


Custo Total Mínimo: 297.0
------------------------------
Matriz 1: 63 chapas
Matriz 4: 117 chapas
Total produzido: 504 médias e 351 grandes


#### Ex4: Problema de Transbordo (Transshipment Problem) /

P1 ==> Node 1
P2 ==> Node 2
D1 ==> Node 5
D2 ==> Node 6
D3 ==> Node 7

In [21]:
m = Model(name="ex4-FabricaAutomovel")

NOS_ORIGEM = [1, 2, 3, 4, 5, 6]
NOS_DESTINO = [3, 4, 5, 6, 7]
x = m.integer_var_dict(
    ((i, j) for i in NOS_ORIGEM for j in NOS_DESTINO),
    lb=0,
    name="qtd_carros_enviados",
)

obj = (
    3 * x[1, 3] + 4 * x[1, 4] + 2 * x[2, 3] + 5 * x[2, 4]
    + 7 * x[3, 4] + 8 * x[3, 5] + 6 * x[3, 6]
    + 4 * x[4, 6] + 9 * x[4, 7] + 5 * x[5, 6] + 3 * x[6, 7]
)
m.minimize(obj)

In [22]:
m.add_constraint(x[1, 3] + x[1, 4] == 1000, ctname="rest_no1")
m.add_constraint(x[2, 3] + x[2, 4] == 1200, ctname="rest_no2")
m.add_constraint(x[3, 4] + x[3, 5] + x[3, 6] == x[1, 3] + x[2, 3], ctname="rest_no3")
m.add_constraint(x[4, 6] + x[4, 7] == x[1, 4] + x[2, 4] + x[3, 4], ctname="rest_no4")
m.add_constraint(x[5, 6] + 800 == x[3, 5], ctname="rest_no5")
m.add_constraint(900 + x[6, 7] == x[3, 6] + x[4, 6] + x[5, 6], ctname="rest_no6")
m.add_constraint(x[4, 7] + x[6, 7] == 500, ctname="rest_no7")

docplex.mp.LinearConstraint[rest_no7](qtd_carros_enviados_4_7+qtd_carros_enviados_6_7,EQ,500)

In [23]:
m.export_as_lp(path="model4_CPLEX.lp")

'model4_CPLEX.lp'

In [24]:
solution = m.solve(log_output=True)

Version identifier: 22.1.1.0 | 2022-11-27 | 9160aff4d
CPXPARAM_Read_DataCheck                          1
Tried aggregator 2 times.
MIP Presolve eliminated 0 rows and 20 columns.
MIP Presolve added 1 rows and 1 columns.
Aggregator did 4 substitutions.
Reduced MIP has 4 rows, 7 columns, and 15 nonzeros.
Reduced MIP has 0 binaries, 7 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.01 sec. (0.03 ticks)
Found incumbent of value 22200.000000 after 0.01 sec. (0.04 ticks)
Tried aggregator 1 time.
MIP Presolve eliminated 1 rows and 1 columns.
MIP Presolve added 1 rows and 1 columns.
Reduced MIP has 4 rows, 7 columns, and 15 nonzeros.
Reduced MIP has 0 binaries, 7 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.00 sec. (0.01 ticks)
MIP emphasis: balance optimality and feasibility.
MIP search method: dynamic search.
Parallel mode: deterministic, using up to 8 threads.
Root relaxation solution time = 0.00 sec. (0.01 ticks)

        Nodes                                         Cuts/
  

In [25]:
if solution:
    print(f"\nCusto Total Mínimo: {m.objective_value}")
    print("-" * 30)
    for chave, var in x.items():
        if var.solution_value > 0:
            print(f"Trajeto {chave} (Origem {chave[0]} -> Destino {chave[1]}): {var.solution_value}")
else:
    print("O modelo não encontrou uma solução ótima.")


Custo Total Mínimo: 20700.0
------------------------------
Trajeto (1, 4) (Origem 1 -> Destino 4): 1000.0
Trajeto (2, 3) (Origem 2 -> Destino 3): 1200.0
Trajeto (3, 5) (Origem 3 -> Destino 5): 800.0
Trajeto (3, 6) (Origem 3 -> Destino 6): 400.0
Trajeto (4, 6) (Origem 4 -> Destino 6): 1000.0
Trajeto (6, 7) (Origem 6 -> Destino 7): 500.0
